# Lab 20 solution, notebook 1: fix the holiday answer without breaking the other three

The worked answer to the challenge of `20a_meet_evals_and_tracing.ipynb`. The setup repeats what the task needs from
the notebook (the opening hours, the helper, the four cases, `run_cases` and the MLflow page), so this file runs on its
own.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
import re
from openai import AsyncOpenAI
from agents import Agent, Runner, ModelSettings, function_tool, set_default_openai_client
from openai.types.shared import Reasoning

set_default_openai_client(AsyncOpenAI(timeout=60, max_retries=2))

HOURS = (
    "Riverside Library opening hours: Monday to Friday 9:00 to 20:00. "
    "Saturday 10:00 to 16:00. Sunday closed."
)


@function_tool
def get_opening_hours() -> str:
    """Read the library's opening hours."""
    return HOURS


INSTRUCTIONS = (
    "You answer questions about the Riverside Library. Use get_opening_hours for opening times. "
    "Answer in one short sentence, and start with Yes or No when the question asks one."
)
helper = Agent(
    name="Library helper",
    instructions=INSTRUCTIONS,
    tools=[get_opening_hours],
    model=MODEL,
    model_settings=ModelSettings(reasoning=Reasoning(effort=EFFORT)),
)

CASES = [
    {"question": "What time do you close on Saturday?", "expect": r"16:00|4(:00)? ?p\.?m"},
    {"question": "Are you open on Sunday?", "expect": r"\bno\b|closed"},
    {"question": "When do you open on Wednesday?", "expect": r"\b9(:00)?\b"},
    {"question": "Are you open on the public holiday next Monday?",
     "expect": r"(don.t|doesn.t|do not|does not|can.t|cannot|not) (say|specify|cover|list|mention|confirm|know)"},
]


async def run_cases(agent, cases):
    for case in cases:
        result = await Runner.run(agent, case["question"])
        passed = bool(re.search(case["expect"], result.final_output, re.IGNORECASE))
        print("✅" if passed else "❌", case["question"], "->", result.final_output)

The MLflow page and recording, as in Part B of the notebook:

In [3]:
import subprocess
import sys
import time
import requests

PORT = 5020
URL = f"http://127.0.0.1:{PORT}"
LAB = ROOT / "20_Evals_And_Tracing"
(LAB / "outputs").mkdir(exist_ok=True)


def page_is_up():
    try:
        return requests.get(URL + "/health", timeout=1).ok
    except requests.ConnectionError:
        return False


server = None
if not page_is_up():
    server = subprocess.Popen(
        [sys.executable, "-m", "mlflow", "server", "--host", "127.0.0.1", "--port", str(PORT),
         "--backend-store-uri", "sqlite:///outputs/mlflow.db"],
        cwd=LAB, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    while not page_is_up():
        assert server.poll() is None, f"The MLflow server stopped. Is port {PORT} used by another program?"
        time.sleep(1)
print("The MLflow page is running at", URL)

The MLflow page is running at http://127.0.0.1:5020


In [4]:
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")
import mlflow

mlflow.set_tracking_uri(URL)
mlflow.set_experiment("Library helper")
mlflow.openai.autolog()
mlflow.tracing.disable_notebook_display()

## Task: one more sentence in the instructions

The new sentence says what to do when the hours do not cover a day or date: say so, instead of guessing.
`helper.clone(instructions=...)` makes a copy of the agent with the new instructions; the original helper is
unchanged. Then all four cases run on the new version.

In [5]:
fixed = helper.clone(
    instructions=INSTRUCTIONS + " If the opening hours do not cover a day or date, say so instead of guessing."
)

await run_cases(fixed, CASES)

✅ What time do you close on Saturday? -> Riverside Library closes at 4:00 p.m. on Saturday.


✅ Are you open on Sunday? -> No, Riverside Library is closed on Sundays.


✅ When do you open on Wednesday? -> Riverside Library opens at 9:00 a.m. on Wednesday.


✅ Are you open on the public holiday next Monday? -> The hours don’t specify whether Riverside Library is open on public holidays, so I can’t confirm for next Monday.


Four ticks. The holiday reply is now *The hours don't specify whether Riverside Library is open on public holidays, so I can't confirm for next Monday.*, and the three other cases still pass. One sentence fixed the failing case, and running the whole set showed it broke nothing else.

The last trace is the holiday question's. Its tool span shows the helper read the same opening hours as before; what
changed is the instructions, and with them the answer.

In [6]:
trace = mlflow.get_trace(mlflow.get_last_active_trace_id(), flush=True)
print("question:", trace.info.request_preview)
print("answer:  ", trace.info.response_preview)
print("tool read:", trace.search_spans(span_type="TOOL")[0].outputs)

question: "Are you open on the public holiday next Monday?"
answer:   "The hours don’t specify whether Riverside Library is open on public holidays, so I can’t confirm for next Monday."
tool read: Riverside Library opening hours: Monday to Friday 9:00 to 20:00. Saturday 10:00 to 16:00. Sunday closed.


The tool returned the same opening hours as in the notebook's run.

On the page (**Experiments**, **Library helper**, **Traces**), the top four rows are this run's four cases. Open the
newest holiday trace, then an older one from the notebook, to compare the two answers.

## Stop the MLflow page

In [7]:
if server is not None:
    server.terminate()
    server.wait()
    print("The MLflow page has stopped.")

The MLflow page has stopped.
